## Sample Dataset Generator

Generates a multi-industry synthetic data catalog in Unity Catalog with **realistic statistical distributions** and Faker-generated PII. Each industry schema contains one **entity table** and one high-volume **event table** (100K–500K rows). All date columns use native Date/Timestamp types.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog. Dropped and recreated on each run. |

### Schemas

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `finance` | `accounts` | \~2K | `transactions` | 100K–500K | 7 account types, 7 currencies, 40+ merchants, pattern-based fraud flags |
| `manufacturing` | `equipment` | \~3K | `production_orders` | 100K–500K | Maintenance tracking, energy consumption, scrap cost, yield variance (70–105%) |
| `retail` | `customers` | \~5K | `orders` | 100K–500K | 60+ products across 7 categories, demographics, product-level return rates |
| `telecom` | `subscribers` | \~5K | `call_records` | 100K–500K | Temporal churn dates, network quality metrics (latency/signal), region-correlated towers |
| `utilities` | `meters` | \~5K | `usage_records` | 100K–500K | Temperature-driven usage, grid zones, spatially correlated outages |
| `health` | `patients` | \~5K | `medical_records` | 100K–500K | Comorbidity modeling (up to 3 conditions), \~35 ICD-10 codes, 50 physicians |
| `gaming` | `players` | \~5K | `wager_records` | 100K–500K | Activity-earned VIP tiers, session tracking, behavior-driven RG flags |
| `insurance` | `policyholders` | \~22K | `claims` | 100K–500K | Multi-policy (15K people), all 50 states + DC, pattern-based fraud, subrogation |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook drops and recreates the catalog, then runs each industry notebook sequentially.
3. Final cells apply a catalog description and a `RemoveAfter` tag for workspace retention compliance.

### How to Extend

To add a new industry, create a new child notebook following the same pattern as the existing ones, then add a `dbutils.notebook.run` call to this master notebook.

In [ ]:
dbutils.widgets.text("catalog", "industry_sample_data")

In [0]:
%sql
DROP CATALOG IF EXISTS IDENTIFIER(:catalog) CASCADE;

CREATE CATALOG IDENTIFIER(:catalog);

In [0]:
import time

CATALOG = dbutils.widgets.get('catalog')

notebooks = [
    "Finance Dataset Generator",
    "Manufacturing Dataset Generator",
    "Retail Dataset Generator",
    "Telecom Dataset Generator",
    "Utilities Dataset Generator",
    "Health Dataset Generator",
    "Gaming Dataset Generator",
    "Insurance Dataset Generator",
]

for nb in notebooks:
    print(f"\u25B6 Running {nb}...")
    start = time.time()
    dbutils.notebook.run(nb, timeout_seconds=3600, arguments={"catalog": CATALOG})
    elapsed = time.time() - start
    print(f"\u2714 {nb} completed in {elapsed:.1f}s\n")

print(f"\u2705 All 8 industry datasets generated in catalog `{CATALOG}`")

▶ Running Finance Dataset Generator...
✔ Finance Dataset Generator completed in 142.4s

▶ Running Manufacturing Dataset Generator...
✔ Manufacturing Dataset Generator completed in 171.6s

▶ Running Retail Dataset Generator...
✔ Retail Dataset Generator completed in 101.3s

▶ Running Telecom Dataset Generator...
✔ Telecom Dataset Generator completed in 151.5s

▶ Running Utilities Dataset Generator...
✔ Utilities Dataset Generator completed in 151.5s

▶ Running Health Dataset Generator...
✔ Health Dataset Generator completed in 141.5s

▶ Running Gaming Dataset Generator...
✔ Gaming Dataset Generator completed in 81.2s

▶ Running Insurance Dataset Generator...
✔ Insurance Dataset Generator completed in 91.2s

✅ All 8 industry datasets generated in catalog `industry_sample_data`


In [0]:
%sql
COMMENT ON CATALOG IDENTIFIER(:catalog) IS
'# Industry Sample Data Catalog\n\nSynthetically generated multi-domain dataset with **realistic statistical distributions** (log-normal, beta, gamma) and **Faker-generated names/emails** for realistic PII. Each schema contains one entity table and one high-volume event table (100K\u2013500K rows). All date columns use native Date/Timestamp types. All tables include primary key and foreign key constraints.\n\n## Schemas\n\n- **finance** \u2014 `accounts` (~2K) + `transactions` (100K\u2013500K): 7 account types, 7 currencies, 40+ merchants, pattern-based fraud flags\n- **gaming** \u2014 `players` (~5K) + `wager_records` (100K\u2013500K): Activity-earned VIP tiers, session tracking, behavior-driven RG flags, reconciled deposits/withdrawals\n- **health** \u2014 `patients` (~5K) + `medical_records` (100K\u2013500K): Comorbidity modeling (up to 3 conditions), ~35 ICD-10 codes, 50 physicians, age derived from DOB\n- **insurance** \u2014 `policyholders` (~22K from 15K people) + `claims` (100K\u2013500K): Multi-policy support, all 50 states + DC, pattern-based fraud, subrogation recovery\n- **manufacturing** \u2014 `equipment` (~3K) + `production_orders` (100K\u2013500K): Maintenance tracking, energy consumption, scrap cost, wider yield variance (70\u2013105%)\n- **retail** \u2014 `customers` (~5K) + `orders` (100K\u2013500K): 60+ products across 7 categories, customer demographics, product-level return rates, smooth seasonal curves\n- **telecom** \u2014 `subscribers` (~5K) + `call_records` (100K\u2013500K): Temporal churn dates, network quality metrics (latency/signal), region-correlated towers\n- **utilities** \u2014 `meters` (~5K) + `usage_records` (100K\u2013500K): Temperature-driven usage, grid zones, spatially correlated outages\n\n> All data is randomized per run. Distributions are parameterized to mimic real-world patterns for analytics, BI, and ML use cases.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER CATALOG `{CATALOG}` SET TAGS ('RemoveAfter' = '{remove_after_value}')")

print(f"✔ RemoveAfter tag applied to catalog {CATALOG}")

✔ RemoveAfter tag applied to catalog and all 8 schemas (2026-12-31)
